In [1]:
import os
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [8]:
DATA_PATH = "../data/processed/consolidated_zip_month.csv"  

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head(3)

(337414, 48)


,zip,month,year,zhvi,zori,MEDIAN_SALE_PRICE,MEDIAN_LIST_PRICE,MEDIAN_PPSF,MEDIAN_LIST_PPSF,HOMES_SOLD,PENDING_SALES,NEW_LISTINGS,INVENTORY,MONTHS_OF_SUPPLY,MEDIAN_DOM,AVG_SALE_TO_LIST,SOLD_ABOVE_LIST,PRICE_DROPS,OFF_MARKET_IN_TWO_WEEKS,pmms30,pmms15,pmms51,pmms51spread,B19013_001E,B25077_001E,B25064_001E,owner_rate,ESTAB,EMP,PAYANN,PAYQTR1,avg_annual_pay_per_emp,N1,A00100,A00200,A00300,A00600,zori_to_zhvi_ratio,zhvi_mom_pct,zhvi_yoy_pct,zori_mom_pct,zori_yoy_pct,MEDIAN_SALE_PRICE_mom_pct,MEDIAN_SALE_PRICE_yoy_pct,INVENTORY_mom_pct,INVENTORY_yoy_pct,MONTHS_OF_SUPPLY_mom_pct,MONTHS_OF_SUPPLY_yoy_pct
0,27006,2000-01-31,2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.210,7.8025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,27006,2000-02-29,2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.325,7.9325,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,27006,2000-03-31,2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.240,7.8320,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Missingness by column (%), measured across ZIPs for each year (2018-2022).
if "df" not in globals():
    DATA_PATH = "../data/processed/consolidated_zip_month.csv"
    df = pd.read_csv(DATA_PATH)

work = df.copy()
work["zip"] = work["zip"].astype(str).str.zfill(5)
work["month"] = pd.to_datetime(work["month"], errors="coerce")
work = work.dropna(subset=["month"])

# Keep only 2018-01 through 2022-12.
work = work[(work["month"] >= "2018-01-01") & (work["month"] <= "2022-12-31")].copy()
work["year"] = work["month"].dt.year

exclude_cols = {"zip", "month", "year"}
feature_cols = [c for c in work.columns if c not in exclude_cols]

# For each (year, zip), mark whether a column has at least one non-null value in that year.
zip_year_has_data = work.groupby(["year", "zip"], as_index=True)[feature_cols].agg(
    lambda s: s.notna().any()
 )

# Convert coverage to missingness percentage by year across ZIPs.
missing_pct_by_year = (1 - zip_year_has_data.groupby(level="year").mean()) * 100
missing_pct_by_year = missing_pct_by_year.sort_index().round(2)

print("Missing % by column for each year (ZIP-based), 2018-2022:")
display(missing_pct_by_year)

# Optional: top 10 most-missing columns per year.
for yr in missing_pct_by_year.index:
    print(f"\nTop 10 most-missing columns in {yr}:")
    display(missing_pct_by_year.loc[yr].sort_values(ascending=False).head(10).to_frame("missing_pct"))

Missing % by column for each year (ZIP-based), 2018-2022:


,zhvi,zori,MEDIAN_SALE_PRICE,MEDIAN_LIST_PRICE,MEDIAN_PPSF,MEDIAN_LIST_PPSF,HOMES_SOLD,PENDING_SALES,NEW_LISTINGS,INVENTORY,MONTHS_OF_SUPPLY,MEDIAN_DOM,AVG_SALE_TO_LIST,SOLD_ABOVE_LIST,PRICE_DROPS,OFF_MARKET_IN_TWO_WEEKS,pmms30,pmms15,pmms51,pmms51spread,B19013_001E,B25077_001E,B25064_001E,owner_rate,ESTAB,EMP,PAYANN,PAYQTR1,avg_annual_pay_per_emp,N1,A00100,A00200,A00300,A00600,zori_to_zhvi_ratio,zhvi_mom_pct,zhvi_yoy_pct,zori_mom_pct,zori_yoy_pct,MEDIAN_SALE_PRICE_mom_pct,MEDIAN_SALE_PRICE_yoy_pct,INVENTORY_mom_pct,INVENTORY_yoy_pct,MONTHS_OF_SUPPLY_mom_pct,MONTHS_OF_SUPPLY_yoy_pct
year,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2018,4.73,89.42,21.61,22.17,21.99,22.45,21.61,31.63,22.08,22.45,100.0,21.61,21.71,21.61,100.0,21.71,0.0,0.0,0.0,0.0,1.39,1.02,7.33,0.37,0.37,0.37,0.37,0.37,0.37,2.32,2.32,2.32,2.32,2.32,89.42,4.73,6.12,89.89,90.91,21.61,22.45,22.54,23.28,100.0,100.0
2019,3.62,88.78,21.34,21.80,21.61,22.08,21.34,31.26,21.80,22.08,100.0,21.34,21.43,21.34,100.0,21.34,0.0,0.0,0.0,0.0,1.11,0.93,6.68,0.37,0.46,0.46,0.46,0.46,0.46,2.32,2.32,2.32,2.32,2.32,88.78,3.62,4.73,88.87,89.42,21.34,22.36,22.45,23.47,100.0,100.0
2020,3.34,87.66,21.24,21.34,21.43,21.52,21.24,31.17,21.34,21.61,100.0,21.24,21.34,21.24,100.0,21.24,0.0,0.0,0.0,0.0,1.48,1.95,7.79,0.37,0.46,0.46,0.46,0.46,0.46,2.23,2.23,2.23,2.23,2.23,87.66,3.34,3.62,87.94,88.78,21.34,21.80,21.99,22.73,100.0,100.0
2021,3.06,85.44,21.06,21.43,21.24,21.52,21.06,30.98,21.43,21.89,100.0,21.06,21.06,21.06,100.0,21.06,0.0,0.0,0.0,0.0,1.48,1.39,8.16,0.09,0.37,0.37,0.37,0.37,0.37,2.32,2.32,2.32,2.32,2.32,85.44,3.06,3.34,85.81,87.66,21.06,21.61,22.08,22.82,100.0,100.0
2022,1.86,81.08,21.15,21.52,21.34,21.61,21.15,31.26,21.52,21.80,100.0,21.15,21.15,21.15,100.0,21.34,0.0,0.0,0.0,0.0,1.76,1.58,7.79,0.09,0.28,0.28,0.28,0.28,0.28,2.32,2.32,2.32,2.32,2.32,81.08,1.86,3.06,81.54,85.44,21.15,21.61,22.08,22.73,100.0,100.0



Top 10 most-missing columns in 2018:


,missing_pct
MONTHS_OF_SUPPLY,100.00
MONTHS_OF_SUPPLY_yoy_pct,100.00
MONTHS_OF_SUPPLY_mom_pct,100.00
PRICE_DROPS,100.00
zori_yoy_pct,90.91
zori_mom_pct,89.89
zori,89.42
zori_to_zhvi_ratio,89.42
PENDING_SALES,31.63
INVENTORY_yoy_pct,23.28



Top 10 most-missing columns in 2019:


...

,missing_pct
MONTHS_OF_SUPPLY,100.00
MONTHS_OF_SUPPLY_yoy_pct,100.00
MONTHS_OF_SUPPLY_mom_pct,100.00
PRICE_DROPS,100.00
zori_yoy_pct,88.78
zori_mom_pct,87.94
zori,87.66
zori_to_zhvi_ratio,87.66
PENDING_SALES,31.17
INVENTORY_yoy_pct,22.73



Top 10 most-missing columns in 2021:


,missing_pct
MONTHS_OF_SUPPLY,100.00
MONTHS_OF_SUPPLY_yoy_pct,100.00
MONTHS_OF_SUPPLY_mom_pct,100.00
PRICE_DROPS,100.00
zori_yoy_pct,87.66
zori_mom_pct,85.81
zori,85.44
zori_to_zhvi_ratio,85.44
PENDING_SALES,30.98
INVENTORY_yoy_pct,22.82



Top 10 most-missing columns in 2022:


,missing_pct
MONTHS_OF_SUPPLY,100.00
MONTHS_OF_SUPPLY_yoy_pct,100.00
MONTHS_OF_SUPPLY_mom_pct,100.00
PRICE_DROPS,100.00
zori_yoy_pct,85.44
zori_mom_pct,81.54
zori,81.08
zori_to_zhvi_ratio,81.08
PENDING_SALES,31.26
INVENTORY_yoy_pct,22.73


In [19]:
import numpy as np
import pandas as pd

DROP_ALWAYS = [
    "MONTHS_OF_SUPPLY", "MONTHS_OF_SUPPLY_mom_pct", "MONTHS_OF_SUPPLY_yoy_pct",
    "PRICE_DROPS"
]

# choose a threshold for "crazy high" missing
MISS_THRESH = 0.60  # 50%+ missing = drop

missing = work.isna().mean().sort_values(ascending=False)
drop_high_missing = missing[missing >= MISS_THRESH].index.tolist()

# combine and keep only what exists
drop_cols = sorted(set([c for c in DROP_ALWAYS if c in work.columns] + drop_high_missing))

print("Dropping columns (count):", len(drop_cols))
print(drop_cols)

df2 = work.drop(columns=drop_cols).copy()
print("New shape:", df2.shape)

Dropping columns (count): 8
['MONTHS_OF_SUPPLY', 'MONTHS_OF_SUPPLY_mom_pct', 'MONTHS_OF_SUPPLY_yoy_pct', 'PRICE_DROPS', 'zori', 'zori_mom_pct', 'zori_to_zhvi_ratio', 'zori_yoy_pct']
New shape: (64680, 40)


In [20]:
KEYS = ["zip", "month"]  # assuming these exist
feature_cols = [c for c in df2.columns if c not in KEYS and c != "year"]

# percent missing per ZIP per feature
miss_by_zip = df2.groupby("zip")[feature_cols].apply(lambda x: x.isna().mean())
# For each feature: how many ZIPs are "bad" (e.g., >35% missing)
BAD_ZIP_THRESH = 0.35

bad_zip_counts = (miss_by_zip > BAD_ZIP_THRESH).sum().sort_values(ascending=False)
bad_zip_pct = (bad_zip_counts / miss_by_zip.shape[0] * 100).round(2)

report = pd.DataFrame({
    "bad_zip_count": bad_zip_counts,
    "bad_zip_pct": bad_zip_pct,
    "overall_missing_pct": (df2[feature_cols].isna().mean()*100).round(2),
}).sort_values(["bad_zip_pct","overall_missing_pct"], ascending=False)

report.head(30)

,bad_zip_count,bad_zip_pct,overall_missing_pct
PENDING_SALES,365,33.86,34.57
INVENTORY_yoy_pct,325,30.15,28.35
INVENTORY_mom_pct,305,28.29,26.98
MEDIAN_SALE_PRICE_yoy_pct,300,27.83,26.81
MEDIAN_LIST_PPSF,290,26.90,26.45
MEDIAN_LIST_PRICE,284,26.35,25.94
NEW_LISTINGS,284,26.35,25.93
INVENTORY,275,25.51,25.72
MEDIAN_SALE_PRICE_mom_pct,273,25.32,25.48
MEDIAN_PPSF,268,24.86,25.09


In [21]:
zip_missing_score = miss_by_zip.mean(axis=1).sort_values(ascending=False)
zip_missing_score.describe()

count    1078.000000
mean        0.123549
std         0.184850
min         0.000901
25%         0.000901
50%         0.009910
75%         0.192342
max         0.748198
dtype: float64

In [22]:
ZIP_DROP_THRESH = 0.35
zips_to_drop = zip_missing_score[zip_missing_score > ZIP_DROP_THRESH].index
print("ZIPs dropped:", len(zips_to_drop), "out of", miss_by_zip.shape[0])

ZIPs dropped: 239 out of 1078


In [23]:
miss_by_year_rows = work.groupby("year")[df2.columns.difference(["zip","month","year"])].apply(lambda x: x.isna().mean()*100).round(2)
miss_by_year_rows

,A00100,A00200,A00300,A00600,AVG_SALE_TO_LIST,B19013_001E,B25064_001E,B25077_001E,EMP,ESTAB,HOMES_SOLD,INVENTORY,INVENTORY_mom_pct,INVENTORY_yoy_pct,MEDIAN_DOM,MEDIAN_LIST_PPSF,MEDIAN_LIST_PRICE,MEDIAN_PPSF,MEDIAN_SALE_PRICE,MEDIAN_SALE_PRICE_mom_pct,MEDIAN_SALE_PRICE_yoy_pct,N1,NEW_LISTINGS,OFF_MARKET_IN_TWO_WEEKS,PAYANN,PAYQTR1,PENDING_SALES,SOLD_ABOVE_LIST,avg_annual_pay_per_emp,owner_rate,pmms15,pmms30,pmms51,pmms51spread,zhvi,zhvi_mom_pct,zhvi_yoy_pct
year,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2018,2.32,2.32,2.32,2.32,25.73,1.39,7.33,1.02,0.37,0.37,25.47,26.28,27.50,29.25,25.60,27.71,27.18,26.25,25.47,26.67,28.20,2.32,27.14,26.09,0.37,0.37,35.80,25.47,0.37,0.37,0.0,0.0,0.00,0.00,4.99,5.12,6.27
2019,2.32,2.32,2.32,2.32,25.00,1.11,6.68,0.93,0.46,0.46,24.68,25.80,27.06,28.46,24.77,27.13,26.47,25.39,24.68,25.83,27.40,2.32,26.45,24.91,0.46,0.46,34.66,24.68,0.46,0.37,0.0,0.0,0.00,0.00,3.71,3.80,4.99
2020,2.23,2.23,2.23,2.23,24.62,1.48,7.79,1.95,0.46,0.46,24.50,25.83,27.17,28.18,24.54,26.61,26.05,25.17,24.50,25.66,26.65,2.23,26.05,24.74,0.46,0.46,34.49,24.50,0.46,0.37,0.0,0.0,0.00,0.00,3.40,3.45,3.73
2021,2.32,2.32,2.32,2.32,23.69,1.48,8.16,1.39,0.37,0.37,23.62,25.29,26.47,28.24,23.66,25.60,25.09,24.27,23.62,24.50,26.24,2.32,25.05,23.96,0.37,0.37,33.69,23.62,0.37,0.09,0.0,0.0,0.00,0.00,3.08,3.11,3.40
2022,2.32,2.32,2.32,2.32,23.93,1.76,7.79,1.58,0.28,0.28,23.83,25.41,26.70,27.61,23.87,25.19,24.93,24.37,23.83,24.72,25.56,2.32,24.93,24.44,0.28,0.28,34.21,23.83,0.28,0.09,0.0,0.0,8.33,8.33,1.96,2.06,3.08


In [24]:
ZIP_DROP_THRESH = 0.35
bad_zips = zip_missing_score[zip_missing_score > ZIP_DROP_THRESH].index

good = df2[~df2["zip"].isin(bad_zips)]
bad  = df2[df2["zip"].isin(bad_zips)]

feature_cols = [c for c in df2.columns if c not in ["zip","month","year"]]

compare = pd.DataFrame({
    "missing_good_pct": (good[feature_cols].isna().mean()*100).round(2),
    "missing_bad_pct": (bad[feature_cols].isna().mean()*100).round(2),
}).assign(diff_pct=lambda x: (x["missing_bad_pct"] - x["missing_good_pct"]).round(2))\
  .sort_values("diff_pct", ascending=False)

compare.head(25)

,missing_good_pct,missing_bad_pct,diff_pct
MEDIAN_SALE_PRICE,3.51,97.80,94.29
SOLD_ABOVE_LIST,3.51,97.80,94.29
HOMES_SOLD,3.51,97.80,94.29
MEDIAN_DOM,3.60,97.83,94.23
OFF_MARKET_IN_TWO_WEEKS,3.97,98.05,94.08
AVG_SALE_TO_LIST,3.74,97.80,94.06
INVENTORY,4.93,98.70,93.77
MEDIAN_PPSF,4.33,97.96,93.63
NEW_LISTINGS,5.19,98.72,93.53
MEDIAN_LIST_PRICE,5.21,98.72,93.51


In [29]:
import numpy as np
import pandas as pd

OUT_PATH = "../data/processed/consolidated_zip_month_redfin_covered_zips.csv"
REMOVED_ZIPS_PATH = "../data/processed/removed_zips_no_redfin_coverage.csv"

# --- 1) Normalize column names (handles invisible spaces / casing issues) ---
df3 = df2.copy()
df3.columns = [c.strip() for c in df3.columns]  # trims whitespace

# --- 2) Find the Redfin columns by case-insensitive match ---
wanted_redfin = [
    "MEDIAN_SALE_PRICE","SOLD_ABOVE_LIST","HOMES_SOLD","MEDIAN_DOM",
    "OFF_MARKET_IN_TWO_WEEKS","AVG_SALE_TO_LIST","INVENTORY","NEW_LISTINGS",
    "MEDIAN_LIST_PRICE","MEDIAN_PPSF","MEDIAN_LIST_PPSF","PENDING_SALES"
]

colmap = {c.upper(): c for c in df3.columns}  # maps UPPER -> actual
REDFIN_COLS = [colmap[c] for c in wanted_redfin if c in colmap]

print("Matched Redfin cols:", REDFIN_COLS)
if len(REDFIN_COLS) == 0:
    raise ValueError(
        "No Redfin columns matched. Print df.columns and check naming. "
        "This prevents filtering everything out."
    )

# --- 3) Compute Redfin coverage per ZIP (mean non-null rate across those cols) ---
# This is robust and fast: compute notna -> groupby -> mean (per zip) -> mean across cols
redfin_cov = (
    df3[REDFIN_COLS].notna()
      .groupby(df3["zip"])
      .mean()
      .mean(axis=1)
)

print("Redfin coverage summary:")
print(redfin_cov.describe())

# --- 4) Choose a threshold that drops the “~98–99% missing” ZIPs ---
# Bad ZIPs typically have coverage near 0.00–0.05; good ZIPs near 0.90+.
# 0.80 is a safe default that should remove the ~239 no-coverage ZIPs without being too strict.
REDFIN_ZIP_THRESH = 0.80

keep_zips = redfin_cov[redfin_cov >= REDFIN_ZIP_THRESH].index
removed_zips = redfin_cov[redfin_cov < REDFIN_ZIP_THRESH].index

df_filtered = df3[df3["zip"].isin(keep_zips)].copy()
df_filtered["redfin_coverage"] = df_filtered["zip"].map(redfin_cov)

print("ZIPs kept:", df_filtered["zip"].nunique(), "out of", df3["zip"].nunique())
print("ZIPs removed:", len(removed_zips))
print("Rows kept:", df_filtered.shape[0], "out of", df3.shape[0])

# --- 5) Save full filtered dataset + removed ZIP list ---
df_filtered.to_csv(OUT_PATH, index=False)
pd.DataFrame({"zip": removed_zips, "redfin_coverage": redfin_cov.loc[removed_zips].values}).to_csv(REMOVED_ZIPS_PATH, index=False)

print("Saved:", OUT_PATH)
print("Saved removed ZIP list:", REMOVED_ZIPS_PATH)

Matched Redfin cols: ['MEDIAN_SALE_PRICE', 'SOLD_ABOVE_LIST', 'HOMES_SOLD', 'MEDIAN_DOM', 'OFF_MARKET_IN_TWO_WEEKS', 'AVG_SALE_TO_LIST', 'INVENTORY', 'NEW_LISTINGS', 'MEDIAN_LIST_PRICE', 'MEDIAN_PPSF', 'MEDIAN_LIST_PPSF', 'PENDING_SALES']
Redfin coverage summary:
count    1078.000000
mean        0.740944
std         0.400575
min         0.000000
25%         0.651042
50%         0.997917
75%         1.000000
max         1.000000
dtype: float64
ZIPs kept: 758 out of 1078
ZIPs removed: 320
Rows kept: 45480 out of 64680
Saved: ../data/processed/consolidated_zip_month_redfin_covered_zips.csv
Saved removed ZIP list: ../data/processed/removed_zips_no_redfin_coverage.csv


In [31]:
import numpy as np
import pandas as pd

# ============================================================
# Drop ZIPs with big consecutive missing gaps across MODEL_COLS
# Uses: df_filtered
# Saves:
#  - ../data/processed/zip_missing_gap_report_model_cols.csv
#  - ../data/processed/consolidated_zip_month_redfin_gap_filtered_model_cols.csv
# ============================================================

ZIP_COL = "zip"
MONTH_COL = "month"

# --- choose a streak cutoff (max consecutive missing months allowed) ---
# If cutoff = 3 => you will DROP ZIPs with any column having 3+ missing months in a row.
STREAK_CUTOFF = 3

# --- choose what columns define "data quality" ---
# 1) Start from all numeric columns (excluding keys)
# 2) Optionally exclude columns you plan to drop anyway (e.g., zori stuff)
EXCLUDE_COLS = {
    "zori", "zori_mom_pct", "zori_yoy_pct", "zori_to_zhvi_ratio",
    "MONTHS_OF_SUPPLY", "MONTHS_OF_SUPPLY_mom_pct", "MONTHS_OF_SUPPLY_yoy_pct",
    "PRICE_DROPS"
}

if "df_filtered" not in globals():
    raise NameError("df_filtered not defined.")

dfq = df_filtered.copy()
dfq.columns = [c.strip() for c in dfq.columns]

# keys exist?
for c in [ZIP_COL, MONTH_COL]:
    if c not in dfq.columns:
        raise ValueError(f"Missing key col '{c}' in df_filtered.")

# normalize keys
dfq[MONTH_COL] = pd.to_datetime(dfq[MONTH_COL], errors="coerce")
dfq[ZIP_COL] = dfq[ZIP_COL].astype(str).str.extract(r"(\d{5})", expand=False).str.zfill(5)
dfq = dfq.dropna(subset=[ZIP_COL, MONTH_COL]).copy()

dfq["ym"] = dfq[MONTH_COL].dt.to_period("M")
dfq = dfq.sort_values([ZIP_COL, "ym"]).reset_index(drop=True)

# build MODEL_COLS = numeric columns you care about
numeric_cols = [c for c in dfq.columns if dfq[c].dtype.kind in "biufc"]
MODEL_COLS = [c for c in numeric_cols if c not in {ZIP_COL, "ym"} and c not in EXCLUDE_COLS]

print("Model cols used for gap checks:", len(MODEL_COLS))
if len(MODEL_COLS) == 0:
    raise ValueError("MODEL_COLS ended up empty; check dtypes / exclusions.")

# helper: longest run of missing (True) in a boolean series
def _max_true_streak(bool_series: pd.Series) -> int:
    if len(bool_series) == 0:
        return 0
    x = bool_series.astype(int)
    run_id = (x != x.shift()).cumsum()
    streaks = x.groupby(run_id).sum()
    return int(streaks.max()) if len(streaks) else 0

# compute per-ZIP max streak per column + overall
zips = dfq[ZIP_COL].unique()
records = []

for z in zips:
    g = dfq[dfq[ZIP_COL] == z].set_index("ym")

    # reindex to continuous months for this ZIP
    full_idx = pd.period_range(g.index.min(), g.index.max(), freq="M")
    g = g.reindex(full_idx)

    # per-column max streaks
    col_max = {}
    for c in MODEL_COLS:
        miss = g[c].isna()
        col_max[c] = _max_true_streak(miss)

    # overall "worst column streak" for the ZIP
    worst_streak = max(col_max.values()) if col_max else 0

    # also measure "any missing among MODEL_COLS" streak
    any_missing = g[MODEL_COLS].isna().any(axis=1)
    any_missing_streak = _max_true_streak(any_missing)

    records.append({
        "zip": z,
        "worst_col_missing_streak": worst_streak,
        "any_missing_streak": any_missing_streak,
        # keep top culprit column for diagnostics
        "worst_col": max(col_max, key=col_max.get) if col_max else None,
        "worst_col_streak": worst_streak
    })

gap_report = pd.DataFrame(records).sort_values(
    ["worst_col_missing_streak", "any_missing_streak"],
    ascending=False
)

print(gap_report.head(10))

# ZIPs to drop based on streak cutoff
zips_to_drop = gap_report.loc[gap_report["worst_col_missing_streak"] >= STREAK_CUTOFF, "zip"]
print(f"ZIPs to drop (>= {STREAK_CUTOFF} consecutive missing months in ANY model col):", len(zips_to_drop))

# filter and save
OUT_REPORT = "../data/processed/zip_missing_gap_report_model_cols.csv"
OUT_DATA   = "../data/processed/consolidated_zip_month_redfin_gap_filtered_model_cols.csv"

gap_report.to_csv(OUT_REPORT, index=False)

df_final = dfq[~dfq[ZIP_COL].isin(set(zips_to_drop))].copy()
df_final.drop(columns=["ym"], inplace=True, errors="ignore")

df_final.to_csv(OUT_DATA, index=False)

print("Saved report:", OUT_REPORT)
print("Saved filtered dataset:", OUT_DATA)
print("Original rows:", dfq.shape[0], "Final rows:", df_final.shape[0])
print("Original ZIPs:", dfq[ZIP_COL].nunique(), "Final ZIPs:", df_final[ZIP_COL].nunique())

Model cols used for gap checks: 39
       zip  worst_col_missing_streak  any_missing_streak      worst_col  worst_col_streak
5    27013                        60                  60  PENDING_SALES                60
6    27014                        60                  60             N1                60
28   27054                        60                  60  PENDING_SALES                60
213  27915                        60                  60             N1                60
216  27920                        60                  60             N1                60
221  27936                        60                  60             N1                60
225  27941                        60                  60    B25064_001E                60
226  27943                        60                  60             N1                60
228  27947                        60                  60    B25064_001E                60
236  27968                        60                  60         